In [ ]:
%pip install -q numpy pandas scipy scikit-learn torch tqdm gensim matplotlib mol2vec fair-esm
%pip install -q rdkit psutil

In [ ]:
import os
import pathlib
import subprocess

!git clone --branch double_all https://github.com/glucose20/Temp.git

In [ ]:
import os
import shutil

# Kaggle tự mount dataset tại /kaggle/input/<dataset-name>
base_dir = '/kaggle/input/pretrain-kd-bind-2'
destination_dir = '/kaggle/working/Temp/data'

# Tạo thư mục đích nếu chưa có
os.makedirs(destination_dir, exist_ok=True)

# Danh sách dataset cần copy
datasets = ['kd_bind']

for name in datasets:
    src = os.path.join(base_dir, name)
    dst = os.path.join(destination_dir, name)
    
    # Nếu thư mục đích đã tồn tại thì xóa để tránh lỗi copytree
    if os.path.exists(dst):
        shutil.rmtree(dst)
    
    shutil.copytree(src, dst)
    print(f"✅ Copied {name} → {destination_dir}")

print("\n🎯 All datasets copied successfully to /kaggle/working/data!")

In [ ]:
%cd /kaggle/working/Temp

In [ ]:
!git pull origin double_all

In [ ]:
import pandas as pd
import os

# Configuration - update these paths for your environment
dataset_file = '/kaggle/input/kd-bind/kd_bind.txt'  # Input dataset file
output_dir = '/kaggle/working/Temp/data/dta-5fold-dataset/kd_bind'  # Output directory
dataset_name = 'kd_bind'  # Dataset name

print(f"{'='*80}")
print(f"Creating drugs.csv and prots.csv for {dataset_name}")
print(f"{'='*80}")
print(f"Input:  {dataset_file}")
print(f"Output: {output_dir}")
print(f"{'='*80}\n")

# Read dataset file
print("Loading dataset...")
df = pd.read_csv(dataset_file, sep=' ', header=None, dtype=str)
df.columns = ['drug_id', 'prot_id', 'drug_smile', 'prot_seq', 'label']
print(f"✓ Loaded {len(df):,} entries")

# Extract unique drugs
print("\nExtracting unique drugs...")
drugs_df = df[['drug_id', 'drug_smile']].drop_duplicates(subset='drug_id')
drugs_df = drugs_df.sort_values('drug_id').reset_index(drop=True)

# Keep the column name as 'drug_smile' to match MyDataset.py expectation
print(f"✓ Found {len(drugs_df):,} unique drugs")
print(f"  Columns: {drugs_df.columns.tolist()}")  # Should show ['drug_id', 'drug_smile']

# Extract unique proteins
print("\nExtracting unique proteins...")
prots_df = df[['prot_id', 'prot_seq']].drop_duplicates(subset='prot_id')
prots_df = prots_df.sort_values('prot_id').reset_index(drop=True)
print(f"✓ Found {len(prots_df):,} unique proteins")
print(f"  Columns: {prots_df.columns.tolist()}")  # Should show ['prot_id', 'prot_seq']

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Save drugs.csv with column name 'drug_smile' (as expected by MyDataset.py)
drugs_file = os.path.join(output_dir, f'{dataset_name}_drugs.csv')
drugs_df.to_csv(drugs_file, index=False)
print(f"\n✓ Saved {drugs_file}")
print(f"  Verify: {pd.read_csv(drugs_file).columns.tolist()}")

# Save prots.csv
prots_file = os.path.join(output_dir, f'{dataset_name}_prots.csv')
prots_df.to_csv(prots_file, index=False)
print(f"✓ Saved {prots_file}")
print(f"  Verify: {pd.read_csv(prots_file).columns.tolist()}")

# Statistics
print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")
print(f"Total entries:          {len(df):,}")
print(f"Unique drugs:           {len(drugs_df):,}")
print(f"Unique proteins:        {len(prots_df):,}")
print(f"\nDrug SMILES length:")
print(f"  Mean:                 {drugs_df['drug_smile'].str.len().mean():.0f} chars")
print(f"  Max:                  {drugs_df['drug_smile'].str.len().max():.0f} chars")
print(f"\nProtein sequence length:")
print(f"  Mean:                 {prots_df['prot_seq'].str.len().mean():.0f} aa")
print(f"  Max:                  {prots_df['prot_seq'].str.len().max():.0f} aa")
print(f"{'='*80}\n")

print("✅ Files created with column name matching MyDataset.py!")
print("   - kd_bind_drugs.csv: ['drug_id', 'drug_smile']")
print("   - kd_bind_prots.csv: ['prot_id', 'prot_seq']")
print("\n🔥 Ready for training! Run train_full.py now.")


In [ ]:
# Fix both train_full.py and MyDataset.py
import re

# ============= Fix train_full.py =============
file_path = '/kaggle/working/Temp/code/train_full.py'
with open(file_path, 'r') as f:
    content = f.read()

# Force string dtype for drug_id and prot_id columns
content = re.sub(
    r"df = pd\.read_csv\(data_file, sep=' ', header=None\)",
    "df = pd.read_csv(data_file, sep=' ', header=None, dtype={0: str, 1: str})",
    content
)
content = re.sub(
    r"drug_df = pd\.read_csv\(hp\.drugs_dir\)",
    "drug_df = pd.read_csv(hp.drugs_dir, dtype={'drug_id': str})",
    content
)
content = re.sub(
    r"prot_df = pd\.read_csv\(hp\.prots_dir\)",
    "prot_df = pd.read_csv(hp.prots_dir, dtype={'prot_id': str})",
    content
)

with open(file_path, 'w') as f:
    f.write(content)

print("✅ Fixed train_full.py - force string dtype for IDs")

# ============= Fix MyDataset.py =============
file_path = '/kaggle/working/Temp/code/MyDataset.py'
with open(file_path, 'r') as f:
    content = f.read()

# Find and replace the problematic section
# The key fix: Skip samples that don't have pretrained embeddings
old_pattern = r"for i, pair in enumerate\(batch_data\):\s+" \
              r"drug_id, prot_id, label = pair\[-3\], pair\[-2\], pair\[-1\]\s+" \
              r"drug_smiles = drug_df\.loc\[drug_df\['drug_id'\] == drug_id, 'drug_smile'\]\.iloc\[0\]\s+" \
              r"prot_seq = prot_df\.loc\[prot_df\['prot_id'\] == prot_id, 'prot_seq'\]\.iloc\[0\]"

new_code = """for i, pair in enumerate(batch_data):        
        drug_id, prot_id, label = pair[-3], pair[-2], pair[-1]
        
        # Convert IDs to strings first
        drug_id_str = str(drug_id)
        prot_id_str = str(prot_id)
        
        # Try int for drug_id (some dicts use int keys)
        try:
            drug_id_int = int(drug_id)
        except (ValueError, TypeError):
            drug_id_int = None
        
        # Check if embeddings exist - SKIP if missing
        has_drug_vec = (drug_id_int in mol2vec_dict["vec_dict"]) if drug_id_int else (drug_id_str in mol2vec_dict["vec_dict"])
        has_prot_vec = prot_id_str in protvec_dict["vec_dict"]
        has_drug_mat = drug_id_str in mol2vec_dict["mat_dict"]
        has_prot_mat = prot_id_str in protvec_dict["mat_dict"]
        
        if not (has_drug_vec and has_prot_vec and has_drug_mat and has_prot_mat):
            continue  # Skip this sample
        
        drug_smiles = drug_df.loc[drug_df['drug_id'] == drug_id, 'drug_smile'].iloc[0]
        prot_seq = prot_df.loc[prot_df['prot_id'] == prot_id, 'prot_seq'].iloc[0]"""

content = re.sub(old_pattern, new_code, content, flags=re.MULTILINE | re.DOTALL)

# Also need to fix the embedding retrieval part
old_retrieval = r"drug_id = str\(drug_id\)\s+" \
                r"prot_id = str\(prot_id\)\s+" \
                r"drug_vec = mol2vec_dict\[\"vec_dict\"\]\[drug_id\]\s+" \
                r"prot_vec = protvec_dict\[\"vec_dict\"\]\[prot_id\]\s+" \
                r"drug_mat = mol2vec_dict\[\"mat_dict\"\]\[drug_id\]\s+" \
                r"prot_mat = protvec_dict\[\"mat_dict\"\]\[prot_id\]"

new_retrieval = """# Get embeddings using appropriate key type
        if drug_id_int and drug_id_int in mol2vec_dict["vec_dict"]:
            drug_vec = mol2vec_dict["vec_dict"][drug_id_int]
        else:
            drug_vec = mol2vec_dict["vec_dict"][drug_id_str]
        
        prot_vec = protvec_dict["vec_dict"][prot_id_str]
        drug_mat = mol2vec_dict["mat_dict"][drug_id_str]
        prot_mat = protvec_dict["mat_dict"][prot_id_str]"""

content = re.sub(old_retrieval, new_retrieval, content, flags=re.MULTILINE)

with open(file_path, 'w') as f:
    f.write(content)

print("✅ Fixed MyDataset.py - skip samples without pretrained embeddings")
print("   This prevents KeyError crashes during training")
print("\n🎯 Ready to train!")


In [ ]:
!python code/train_full.py \
  --dataset kd_bind \
  --data_root /kaggle/input/kd-bind \
  --epochs 200 \
  --batch_size 16 \
  --cuda 0